# Initial Route Set Generation

In [1]:
%load_ext autoreload
%reload_ext autoreload

import copy
import os
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import geopandas as gpd
import networkx as nx
import osmnx as ox
import utm
from geopy.distance import geodesic
from networkx import MultiDiGraph
from shapely.geometry import LineString, Point
from shapely.ops import split

from config.config import FILTER_OPTIONS, MAX_DISTANCE, WALKING_DISTANCES
from genetic_algorithm.reverse_routes import get_reverse_route
from genetic_algorithm.schema import Network, Route, Stop
from utils.checkpoint_utils import load_data, save_data
from utils.graph_utils import generate_graph, load_graph, save_route_graph_to_plot
from utils.route_utils import (
    add_points_to_graph,
    degrees_to_meters,
    disable_surrounding_nodes,
    enable_nodes,
    get_enabled_node_with_highest_edge_probability,
    meters_to_degrees,
)


## Graph and Features

In [2]:
# NOTE: SELECT THE CITY HERE, COMMENT OUT THE REMAINING CITIES
CITY_NAME = "Manila" # Taguig, Mandaluyong, Manila
COUNTRY = "Philippines"
MODE = "drive"

LOCATION = f"{CITY_NAME}, {COUNTRY}"
TARGET_MAP_DIR = "../data/street_networks"
GRAPH_DIR = os.path.join(TARGET_MAP_DIR, f"{CITY_NAME}.graphml")

# GENERATION OF MAIN CITY GRAPH
# IF FIRST TIME RUNNING, RUN THIS CODE TO GENERATE THE GRAPH
# def generate_graph():
#     print(f"Generating road network for {LOCATION}")
#     graph = ox.graph_from_place(LOCATION, network_type=MODE)
#     ox.save_graphml(graph, GRAPH_DIR) # Save it as a file

#     print(f"Graph saved to {GRAPH_DIR}")
#     return graph

# def load_graph():
#     graph = ox.load_graphml(GRAPH_DIR)
#     print(f"Graph loaded successfully from {GRAPH_DIR}")

#     return graph

# THIS IS THE MAIN GRAPH FOR THE CITY TO BE USED FOR ALL FUNCTIONS
if os.path.exists(GRAPH_DIR):
    CITY_GRAPH = load_graph(GRAPH_DIR)
else:
    CITY_GRAPH = generate_graph(LOCATION, MODE, GRAPH_DIR)

print("No. of edges: ", CITY_GRAPH.number_of_edges())
print("No. of nodes: ", CITY_GRAPH.number_of_nodes())

# Iterate over the edges and assign a unique edge_id
for edge_id, (u, v, k) in enumerate(CITY_GRAPH.edges(keys=True)):
    CITY_GRAPH[u][v][k]["edge_id"] = edge_id

Graph loaded successfully
No. of edges:  12739
No. of nodes:  5062


## Initial Filtering

In [3]:
### For Filtering the roads and other features

# GETTING ROADS AND WATERWAYS
road = ox.graph_to_gdfs(CITY_GRAPH, nodes=False, edges=True)

# Get all the roads that are not junctions (ex. Roundabouts, intersection, etc.)
filtered_roads = road[road['junction'].isna()]

# Separate roads whose highway types are only one value and those that are more than 1 (lists)
rows_with_lists = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, list))]
rows_with_strings = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, str))]

# Allowed Roads to place stops
filter_options = FILTER_OPTIONS

# To separate zones
separation_options = ['primary', 'secondary', 'tertiary', 'unclassified']

# Get the roads whose widths are above the threshold
def check_list(lst):
    return any(x in filter_options for x in lst)

# Download OpenStreetMap data for the area of interest
waterways = ox.features_from_place(LOCATION, tags={'waterway': True})
filtered_rivers = waterways[waterways['waterway'].isin(['river'])]
filtered_streams = waterways[waterways['waterway'].isin(['stream'])]

# Get all the roads with the allowed road types
filtered_roads_strings = rows_with_strings.loc[rows_with_strings['highway'].isin(filter_options)]
filtered_roads_lists = rows_with_lists[rows_with_lists['highway'].apply(check_list)]

# Create spatial index
filtered_roads_strings_sindex = filtered_roads_strings.sindex
filtered_roads_lists_sindex = filtered_roads_lists.sindex
filtered_rivers_sindex = filtered_rivers.sindex
filtered_streams_sindex = filtered_streams.sindex

# Total Distance of the allowed road types
TOTAL_ROAD_NETWORK_DISTANCE = 0
for index, row in filtered_roads_strings.iterrows():
    TOTAL_ROAD_NETWORK_DISTANCE += row['length']

## Stop Placement


In [ ]:
# CREATING STOPS
# It should return a list of coordinates/nodes for stop and a graph of stops
# if residential area, check if the population density
def place_stops_on_roads(amenity_graph, graph_of_stops, list_of_stops):
    list_relevant_edges_set = []
    string_relevant_edges_set = []

    stop_id = 0
    node_sizes = {
        n: a for n, a in zip(
            amenity_graph.nodes,
            gpd.GeoSeries([d['geometry'] for _, d in amenity_graph.nodes(data=True)],
                          crs="EPSG:4326").to_crs(32651).area,
        )
    }
    for node_key, node_data in amenity_graph.nodes(data=True):

        # All tranportation points are automatically added as stops
        if node_data['geometry'].geom_type in ['Point']:
            if node_data['amenity'] == 'transportation':
                # Find the nearest edge to the location point
                nearest_edge = ox.distance.nearest_edges(graph_of_stops, X=node_data['x'], Y=node_data['y'], return_dist=False)
                node1, node2, key = nearest_edge
                edge = graph_of_stops.get_edge_data(node1, node2)
                data = next(iter(edge.values()))

                # Get the line
                if 'geometry' not in data:
                    gdf_edges = ox.graph_to_gdfs(graph_of_stops, nodes=False, edges=True)
                    line = gdf_edges.loc[nearest_edge]['geometry']
                    edge_length = gdf_edges.loc[nearest_edge]['length']
                else:
                    line = data['geometry']
                    edge_length = data['length']

                if edge_length > 20:
                    # x - lon - 120
                    # y - lat - 14
                    # LineString(lon, lat)
                    # geodesic(lat, lon)

                    print(f"STOP ID: {stop_id}")
                    if node1 < 10000:
                        notice = '!'
                        print(f"SEEN ADDED NODE1{notice} {node1} - {graph_of_stops.nodes[node1]['y']}, {graph_of_stops.nodes[node1]['x']}")

                    if node2 < 10000:
                        notice = '!'
                        print(f"SEEN ADDED NODE2{notice} {node2} - {graph_of_stops.nodes[node2]['y']}, {graph_of_stops.nodes[node2]['x']}")

                    print(f"EDGE NODES ({node1}, {node2})")

                    point_a = (graph_of_stops.nodes[node1]['y'], graph_of_stops.nodes[node1]['x'])
                    point_c = (graph_of_stops.nodes[node2]['y'], graph_of_stops.nodes[node2]['x'])

                    if len(line.coords) == 2:
                        print("Transpo Stop WAY 1")
                        # Get a random position between the line
                        random_position = random.uniform(0.3 * edge_length, edge_length * 0.7)

                        # Convert to UTM
                        node1_utm = utm.from_latlon(point_a[0], point_a[1])
                        node2_utm = utm.from_latlon(point_c[0], point_c[1])

                        utm_line = LineString([(node1_utm[0], node1_utm[1]), (node2_utm[0], node2_utm[1])])

                        # Calculate the coordinate along the edge at the random position
                        point_on_road_utm = calculate_coordinate_along_edge(utm_line, random_position)
                        to_convert = (point_on_road_utm[0], point_on_road_utm[1], node1_utm[2], node1_utm[3])
                        point_on_road = utm.to_latlon(*to_convert)
                        lon = point_on_road[1]
                        lat = point_on_road[0]
                    else:
                        print("Transpo Stop WAY 2")
                        # Accessing all coordinates
                        coordinates = list(line.coords)
                        # Randomly pick one coordinate and split there
                        split_idx = random.randint(1, len(coordinates) - 2)
                        point_on_road = coordinates[split_idx] # coords in shapely order, theyre not flipped so dont edit
                        lon = point_on_road[0] 
                        lat = point_on_road[1]
                        
                        # coordinates = coordinates[1:-1]
                        # # Randomly pick one coordinate
                        # point_on_road = random.choice(coordinates)
                        # lon = point_on_road[0]
                        # lat = point_on_road[1]

                    isTranspo = True

                    graph_of_stops.add_node(stop_id, x=lon, y=lat, isTranspo=isTranspo)
                    list_of_stops.append(Stop(lat, lon, True, stop_id, 0))

                    # Get the points
                    point_b = (lat, lon) # y,x

                    # Calculate new geometries
                    if len(line.coords) == 2:
                        line_ab = LineString([(point_a[1], point_a[0]), (point_b[1], point_b[0])])
                        line_bc = LineString([(point_b[1], point_b[0]), (point_c[1], point_c[0])])
                    else: # use the split to make new lines
                        line_ab = LineString(coordinates[:split_idx + 1])
                        line_bc = LineString(coordinates[split_idx:])
                        
                        # result = split(line, Point(point_b[1], point_b[0]))
                        # new_lines = [geom for geom in result.geoms]
                        # line_ab = new_lines[0]
                        # line_bc = new_lines[1]

                    print("POINT B: ", point_b)

                    print("NEW LINES")
                    print(line_ab)
                    print(line_bc)
                    print(f"ORIGINAL LINE: {line}")
                    print("---------")

                    # Calculate the new distances
                    distance_ab = geodesic(point_a, point_b).meters
                    distance_bc = geodesic(point_b, point_c).meters

                    print("NEW DISTANCES")
                    print(distance_ab)
                    print(distance_bc)
                    print(f"ORIGINAL DISTANCE: {edge_length}")
                    print("---------")


                    # Get the edge data and adjust distances
                    edge_data = data.copy()
                    edge_data['length'] = distance_ab
                    edge_data['geometry'] = line_ab
                    graph_of_stops.add_edge(node1, stop_id, **edge_data)

                    edge_data = data.copy()
                    edge_data['length'] = distance_bc
                    edge_data['geometry'] = line_bc
                    graph_of_stops.add_edge(stop_id, node2, **edge_data)

                    print(f"HAS EDGE BETWEEN {node1} and {stop_id} : {graph_of_stops.has_edge(node1, stop_id)}")
                    print(f"HAS EDGE BETWEEN {stop_id} and {node2} : {graph_of_stops.has_edge(stop_id, node2)}")

                    # Remove the original edge
                    graph_of_stops.remove_edge(node1, node2)
                    stop_id += 1
                    print()


        elif node_data['geometry'].geom_type in ['MultiPolygon', 'Polygon']:
            # Get the roads surrounding and inside the node polygons
            buffer_poly = node_data['geometry'].buffer(meters_to_degrees(30))
            relevant_edges = get_relevant_edges(buffer_poly, list_relevant_edges_set, string_relevant_edges_set)

            # Calculate the number of stops based on node size and population density
            node_size = node_sizes[node_key]

            # Place stops randomly on these roads
            stop_id = place_stops_along_edges(relevant_edges, buffer_poly, node_size, graph_of_stops, list_of_stops, stop_id)

def get_relevant_edges(polygon, list_relevant_edges_set, string_relevant_edges_set):
    relevant_edges = []

    # Check intersection with filtered roads
    possible_matches_roads = filtered_roads_strings.iloc[list(filtered_roads_strings_sindex.intersection(polygon.bounds))]
    for index, row in possible_matches_roads.iterrows():
        if polygon.intersects(row['geometry']) and row['highway'] in filter_options:
            row_edge_id = row['edge_id']
            if row_edge_id not in string_relevant_edges_set:
                relevant_edges.append([index, row])
                string_relevant_edges_set.append(row_edge_id)


    possible_matches_lists = filtered_roads_lists.iloc[list(filtered_roads_lists_sindex.intersection(polygon.bounds))]
    for index, row in possible_matches_lists.iterrows():
        if polygon.intersects(row['geometry']):
            list_highway = row['highway']
            if any(x in filter_options for x in list_highway):
                row_edge_id = row['edge_id']
                if row_edge_id not in list_relevant_edges_set:
                    relevant_edges.append([index, row])
                    list_relevant_edges_set.append(row_edge_id)

    return relevant_edges

def place_stops_along_edges(edges, polygon, node_size, graph_of_stops, list_of_stops, stop_id):
    # Place stops randomly along the edges within the polygon

    if len(edges) > 0:
        for edge in edges:
            # Getting the edge data
            index = edge[0]
            data = edge[1]
            node1 = index[0]
            node2 = index[1]


            # get the line
            if 'geometry' not in data:
                gdf_edges = ox.graph_to_gdfs(graph_of_stops, nodes=False, edges=True)
                line = gdf_edges.loc[(node1, node2, 0)]['geometry']
                edge_length = gdf_edges.loc[(node1, node2, 0)]['length']
            else:
                line = data['geometry']
                edge_length = data['length']

            if edge_length > 20:
                # x - lon - 120
                # y - lat - 14
                # LineString(lon, lat)
                # geodesic(lat, lon)
                print(f"Stop ID: {stop_id}")

                if node1 < 10000:
                        notice = '!'
                        print(f"SEEN ADDED NODE1{notice} ({node1}, {node2}) - {graph_of_stops.nodes[node1]['y']}, {graph_of_stops.nodes[node1]['x']}")

                if node2 < 10000:
                    notice = '!'
                    print(f"SEEN ADDED NODE2{notice} ({node1}, {node2}) - {graph_of_stops.nodes[node2]['y']}, {graph_of_stops.nodes[node2]['x']}")


                # Get the coordinates of the two nodes
                point_a = (graph_of_stops.nodes[node1]['y'], graph_of_stops.nodes[node1]['x'])
                point_c = (graph_of_stops.nodes[node2]['y'], graph_of_stops.nodes[node2]['x'])

                if len(line.coords) == 2:
                    print("Along edges WAY 1")
                    # Get a random position between the line
                    random_position = random.uniform(0.3 * edge_length, edge_length * 0.7)

                    # Convert to UTM
                    node1_utm = utm.from_latlon(point_a[0], point_a[1])
                    node2_utm = utm.from_latlon(point_c[0], point_c[1])

                    utm_line = LineString([(node1_utm[0], node1_utm[1]), (node2_utm[0], node2_utm[1])])

                    # Calculate the coordinate along the edge at the random position
                    point_on_road_utm = calculate_coordinate_along_edge(utm_line, random_position)
                    to_convert = (point_on_road_utm[0], point_on_road_utm[1], node1_utm[2], node1_utm[3])
                    point_on_road = utm.to_latlon(*to_convert)
                    lon = point_on_road[1]
                    lat = point_on_road[0]
                else:
                    print("Along edges WAY 2")
                    # Accessing all coordinates
                    coordinates = list(line.coords)
                    # Randomly pick one coordinate and split there
                    split_idx = random.randint(1, len(coordinates) - 2)
                    point_on_road = coordinates[split_idx] # coords in shapely order, theyre not flipped so dont edit
                    lon = point_on_road[0]
                    lat = point_on_road[1]

                    # coordinates = coordinates[1:-1]
                    # # Randomly pick one coordinate
                    # point_on_road = random.choice(coordinates)
                    # lon = point_on_road[0]
                    # lat = point_on_road[1]

                # Add the stop to graph and list
                isTranspo = False

                graph_of_stops.add_node(stop_id, x=lon, y=lat, isTranspo=isTranspo)
                list_of_stops.append(Stop(lat, lon, False, stop_id, node_size))

                # Get the points
                point_b = (lat, lon)

                # Calculate new geometries

                if len(line.coords) == 2:
                    line_ab = LineString([(point_a[1], point_a[0]), (point_b[1], point_b[0])])
                    line_bc = LineString([(point_b[1], point_b[0]), (point_c[1], point_c[0])])
                else:
                    line_ab = LineString(coordinates[:split_idx + 1])
                    line_bc = LineString(coordinates[split_idx:])
                    # result = split(line, Point(point_b[1], point_b[0]))
                    # new_lines = [geom for geom in result.geoms]
                    # line_ab = new_lines[0]
                    # line_bc = new_lines[1]

                print("POINT B: ", point_b)

                print("NEW LINES")
                print(line_ab)
                print(line_bc)
                print("---------")

                # Calculate the new distances
                distance_ab = geodesic(point_a, point_b).meters
                distance_bc = geodesic(point_b, point_c).meters

                print("NEW DISTANCES")
                print(distance_ab)
                print(distance_bc)
                print(f"ORIGINAL DISTANCE: {data['length']}")
                print("---------")

                # Add the new distances and new geometries
                edge_data = data.copy()
                edge_data['length'] = distance_ab
                edge_data['geometry'] = line_ab
                graph_of_stops.add_edge(node1, stop_id, **edge_data)

                edge_data = data.copy()
                edge_data['length'] = distance_bc
                edge_data['geometry'] = line_bc
                graph_of_stops.add_edge(stop_id, node2, **edge_data)

                print(f"HAS EDGE BETWEEN {node1} and {stop_id} : {graph_of_stops.has_edge(node1, stop_id)}")
                print(f"HAS EDGE BETWEEN {stop_id} and {node2} : {graph_of_stops.has_edge(stop_id, node2)}")

                # Remove the original edge
                graph_of_stops.remove_edge(node1, node2)
                stop_id += 1
                print()

    return stop_id


def calculate_coordinate_along_edge(edge, position):
    # Calculate the coordinate along the edge at the given position
    point = edge.interpolate(position)
    return point.x, point.y


## Route Connection

In [5]:
# V2 - Graph with the snapping function
# Generate Route Network from connected routes
def generate_route_network(stop_nodes: list[Stop], max_walking_dist: int, max_stops: int, max_lines: int, graph_of_stops, city_area_sum: float):
    net_start_time = time.perf_counter()
    nodes_yx = {n: (data['y'], data['x']) for n, data in graph_of_stops.nodes(data=True)}

    overall_graph = nx.MultiDiGraph() # The route network graph
    route_network = []

    num_lines = 0 # Line counter
    max_attempts, attempts = max_lines * 10, 0  # Prevent infinite loop if valid lines cannot be formed

    print(f"Target: {max_lines} lines | Max attempts: {max_attempts}")
    while num_lines < max_lines and attempts < max_attempts:
        line_id = num_lines
        forward_route_id = f"{line_id}-F"

        next_nodes = [n for n in stop_nodes] # Resets the list of nodes so that nodes can be reused in a different
        selected_node = random.choice(next_nodes) # For the first node
        next_nodes.remove(selected_node)

        # Generate forward route
        route_gen, route_length = generate_route(
            selected_node, next_nodes, max_walking_dist, max_stops,
            overall_graph, city_area_sum, forward_route_id, graph_of_stops, nodes_yx
        )

        if len(route_gen) > 1:
            forward_route = Route(route_gen, forward_route_id, route_length, line_id, direction="forward") # Create a new route object to store the Route ID and the route itself

            # Snap the forward route to the graph
            add_to_graph(route_gen, overall_graph, graph_of_stops, forward_route_id, nodes_yx)
            route_network.append(forward_route)

            # Get the reverse route and add it to the graph
            reverse_route = get_reverse_route(forward_route, graph_of_stops, nodes_yx)

            if reverse_route is not None:
                add_to_graph(reverse_route.route, overall_graph, graph_of_stops, reverse_route.route_id, nodes_yx)
                route_network.append(reverse_route)
                print(f"  └─ Line {line_id} COMPLETE: Forward ({route_length:.2f} km) + Reverse ({reverse_route.length:.2f} km)")
            else:
                print(f"  └─ Line {line_id} PARTIAL: Forward added ({route_length:.2f} km), Reverse FAILED")

            num_lines += 1

        else:
            print(f"  [Attempt {attempts}] Failed to generate valid forward route for Line {line_id}. Retrying...")

    net_elapsed = time.perf_counter() - net_start_time
    print(f"Finished: {num_lines}/{max_lines} lines created in {attempts} attempt(s). Done in {net_elapsed:.2f}s.")
    return route_network, overall_graph

def add_to_graph(route, overall_graph, graph_of_stops, route_id, nodes_yx):
    # Directly add nodes based on node identifiers
    for connection in route:
        u_node = connection[0]
        v_node = connection[-1]

        overall_graph.add_node(u_node, **graph_of_stops.nodes[u_node]) # The origin
        overall_graph.add_node(v_node, **graph_of_stops.nodes[v_node]) # The destination

        distance_travelled = 0
        # Get the total distance from point A to point B
        for i in range(len(connection) - 1):
            ay, ax = nodes_yx[connection[i]]
            by, bx = nodes_yx[connection[i+1]]

            distance_travelled += geodesic((ay, ax), (by, bx)).kilometers

        # Finally, Add the edge
        overall_graph.add_edge(u_node, v_node, key=route_id, road_path=connection, distance=distance_travelled)

# Generate route from stop nodes
def generate_route(source: Stop, next_nodes: list[Stop], max_walking_dist: int, max_stops: int, network_graph: MultiDiGraph, city_area_sum: float, route_id: str, graph_of_stops: nx.MultiDiGraph, nodes_yx: dict):
    short_route_list = [] # List of nx.shortest_path results
    total_distance = 0.0
    orig_node = source
    num_stops = 0 # Count number of stops

    # Configuration
    max_tries = 5 # This is the max number of tries before breaking the loop || To avoid longer runtimes
    current_tries = 0

    while total_distance < MAX_DISTANCE and num_stops < max_stops:
        enable_nodes(next_nodes)
        disable_surrounding_nodes(next_nodes, orig_node, max_walking_dist)
        enabled_nodes = [n for n in next_nodes if n.enabled]
        if len(enabled_nodes) == 0:
            break

        # print(f"{len(enabled_nodes)} nodes out of {len(next_nodes)}")

        # Getting the destination node
        dest_node = get_enabled_node_with_highest_edge_probability(orig_node, enabled_nodes, city_area_sum, network_graph)

        if (dest_node == None or dest_node.stop_id == orig_node.stop_id):
            break

        # Remove it as a candidate
        next_nodes.remove(dest_node)

        # This is to check if there is already an exiting edge in the route. If true, then it should not connect
        connection_edge1 = network_graph.has_edge(orig_node.stop_id, dest_node.stop_id, route_id)
        connection_edge2 = network_graph.has_edge(dest_node.stop_id, orig_node.stop_id, route_id)

        # If there is already an existing edge in either direction for this route_id, do not connect
        if connection_edge1 or connection_edge2:
            current_tries += 1
            if current_tries == max_tries:
                break
            continue  # important: skip trying shortest path

        # Try to get shortest path (cached). If no path exists, treat like nx.has_path == False.
        try:
            shortest_route = nx.shortest_path(graph_of_stops, orig_node.stop_id, dest_node.stop_id, weight='length')
        except nx.NetworkXNoPath:
            current_tries += 1
            if current_tries == max_tries:
                break
            continue

        # Get the total distance from point A to point B
        distance_travelled = 0
        for i in range(len(shortest_route)-1):
           ay, ax = nodes_yx[shortest_route[i]]
           by, bx = nodes_yx[shortest_route[i+1]]
           distance_travelled += geodesic((ay, ax), (by, bx)).kilometers

        # Checks if it does not exceed the max distance
        if total_distance + distance_travelled <= MAX_DISTANCE:
            # Updating local degree count used for connection probability
            orig_node.degree += 1
            dest_node.degree += 1

            total_distance += distance_travelled
            short_route_list.append(shortest_route)
            num_stops += 1

            orig_node = dest_node # Now change the origin to the destination
        else:
            break

    if len(short_route_list) > 4 and total_distance > 7:
        return short_route_list, total_distance

    else:
        return [], []

In [6]:
AMENITY_MAP_DIR = f"../data/amenity_maps/Jacob-Group/{CITY_NAME} City"
POLY_FILE = os.path.join(AMENITY_MAP_DIR, f"{CITY_NAME}_polygon.geojson")

merged_amenities_polygons_gdf= gpd.read_file(POLY_FILE)

merged_amenities_polygons_gdf['amenity_points'] = None
merged_amenities_polygons_gdf['area'] = merged_amenities_polygons_gdf.to_crs(epsg=32651).geometry.area

city_area_sum = merged_amenities_polygons_gdf['area'].sum()

In [7]:
SAVED_NETWORKS_DIR = f"../data/saved_networks/{CITY_NAME}"

INITIAL_NETWORK_FILE = os.path.join(SAVED_NETWORKS_DIR, f"{CITY_NAME}_Initial_Network.pkl")
ZONE_NETWORK_FILE = os.path.join(SAVED_NETWORKS_DIR, f"{CITY_NAME}_Zone_Network.pkl")

initial_network = load_data(INITIAL_NETWORK_FILE)
graph_networks_of_polygons = load_data(ZONE_NETWORK_FILE)

add_points_to_graph(initial_network, graph_networks_of_polygons) # Add all points

In [8]:
# LEVEL 3 - CREATING STOPS TO BE PLACED ON ZONES
graph_of_stops = copy.deepcopy(CITY_GRAPH)
list_of_stops = []
place_stops_on_roads(graph_networks_of_polygons, graph_of_stops, list_of_stops) # Adds stops to graph_of_stops, and gets the relevant edges

nid = len(list_of_stops)
if nid in graph_of_stops and graph_of_stops.nodes[nid] == {}:
    graph_of_stops.remove_node(nid)

Stop ID: 0
Along edges WAY 2
POINT B:  (14.5781215, 120.9781886)
NEW LINES
LINESTRING (120.9779706 14.5784948, 120.9781886 14.5781215)
LINESTRING (120.9781886 14.5781215, 120.9784756 14.5776301, 120.978538 14.5775174, 120.9785736 14.5774531, 120.9792208 14.5763093, 120.9795983 14.5755992, 120.9796753 14.5754543, 120.9799131 14.5750182, 120.9807414 14.5734225, 120.9809511 14.5730182)
---------
NEW DISTANCES
47.51666884302556
638.317924772114
ORIGINAL DISTANCE: 688.3975123691496
---------
HAS EDGE BETWEEN 1020279307 and 0 : True
HAS EDGE BETWEEN 0 and 26032777 : True

Stop ID: 1
Along edges WAY 1
POINT B:  (np.float64(14.575930622498502), np.float64(120.97957905778746))
NEW LINES
LINESTRING (120.9800499 14.5750983, 120.97957905778746 14.575930622498502)
LINESTRING (120.97957905778746 14.575930622498502, 120.9793335 14.5763647)
---------
NEW DISTANCES
105.1440264402189
54.83534740247435
ORIGINAL DISTANCE: 160.54097496561184
---------
HAS EDGE BETWEEN 243973011 and 1 : True
HAS EDGE BETWEE

In [ ]:
# Configuration
ROUTE_AMOUNT = {"Makati":28, "Manila":50, "Taguig":18, "Mandaluyong":10} # og routes count (manila capped at 50 for runtime concerns)

NUM_OF_NETWORKS = 100 # ifl these should be hyperparams
MAX_STOPS = [x for x in range(20, 25)]
MAX_LINES = [x for x in range(ROUTE_AMOUNT[CITY_NAME] - 5, ROUTE_AMOUNT[CITY_NAME] + 5)]

SEED = 100
# WALKING_DISTANCES ->  500, 750, 1000 (changed from 300,550,800)

# def make_run_tag():
#   return (f"w{int(WALK_DISTANCE)}_s{MAX_STOPS}_l{MAX_LINES}_n{NUM_OF_NETWORKS}")

# run_tag = make_run_tag()

TARGET_GEN_DIR = f"../data/initial_route_sets/{CITY_NAME}"
os.makedirs(TARGET_GEN_DIR, exist_ok=True)

# file_path = os.path.join(TARGET_GEN_DIR, f"{CITY_NAME}_population_{run_tag}.pkl")
file_path = os.path.join(TARGET_GEN_DIR, f"{CITY_NAME}_population.pkl")

def worker(i):
  rand = random.Random(SEED + i)
  route_network, route_graph = generate_route_network(
      list_of_stops,
      rand.choice(WALKING_DISTANCES),
      rand.choice(MAX_STOPS),
      rand.choice(MAX_LINES),
      graph_of_stops,
      city_area_sum
  )
  return Network(routes=route_network, graph=route_graph)

initial_population = []

with ThreadPoolExecutor() as exec:
  futures = [exec.submit(worker, i) for i in range(NUM_OF_NETWORKS)]

  for idx, future in enumerate(as_completed(futures), 1):
    net = future.result()
    initial_population.append(net)

save_data(initial_population, file_path)
print(f"Saved {len(initial_population)} networks to {file_path}.")

Target: 52 lines | Max attempts: 520Target: 53 lines | Max attempts: 530

Target: 50 lines | Max attempts: 500
Target: 48 lines | Max attempts: 480
Target: 48 lines | Max attempts: 480
Target: 54 lines | Max attempts: 540
Target: 45 lines | Max attempts: 450
Target: 54 lines | Max attempts: 540
Target: 51 lines | Max attempts: 510
Target: 52 lines | Max attempts: 520
Target: 48 lines | Max attempts: 480
Target: 52 lines | Max attempts: 520
Target: 54 lines | Max attempts: 540
Target: 45 lines | Max attempts: 450
Target: 46 lines | Max attempts: 460
Target: 46 lines | Max attempts: 460
Target: 53 lines | Max attempts: 530
Target: 47 lines | Max attempts: 470
Target: 49 lines | Max attempts: 490
Target: 53 lines | Max attempts: 530
  [Attempt 0] Failed to generate valid forward route for Line 0. Retrying...
  [Attempt 0] Failed to generate valid forward route for Line 0. Retrying...
  [Attempt 0] Failed to generate valid forward route for Line 0. Retrying...
  [Attempt 0] Failed to gener

In [ ]:
# file_path = os.path.join(TARGET_GEN_DIR, f"{CITY_NAME}_stops_{run_tag}.pkl")
file_path = os.path.join(TARGET_GEN_DIR, f"{CITY_NAME}_stops.pkl")

save_data(graph_of_stops, file_path)
print(f"Saved stop graph to {file_path}.")

Saved stop graph to ../data/initial_route_sets/Makati/Makati_stops_w300_s20_l25_n50.pkl.


### Visualization of Initial Population

In [ ]:
# Configuration
OUTPUT_DIR = f"../plots/Initial Route Set Maps/{CITY_NAME}/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

route_network = initial_population[0].routes

save_route_graph_to_plot(graph_of_stops, route_network, f"{CITY_NAME}_Combined_Routes.html", OUTPUT_DIR)

Saved combined Makati route map to: ../plots/Initial Route Set Maps/Makati/Makati_Combined_Routes_w300_s20_l25_n50.html


### Visualization of Stops

In [ ]:
# stops_map = plot_stops_on_map(list_of_stops_Makati)
# stops_map.save(f"{Makati_map_filepath}stops_map.html")